# Intro

**Seminar by @Elad_Benjo**

In this part of EDA I will inspect our monthly indicators in order to identify patterns or hidden insights that can use as a lead for the Reduciton stage.

In [1]:
from google.colab import drive
import sys
import pandas as pd

drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/gdp_nowcasting_seminar/src')

Mounted at /content/drive


In [2]:
merged_df = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/stationary/full_data.pkl')

# EDA

In [3]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 308 entries, 1995-01-01 to 2025-04-01
Data columns (total 61 columns):
 #   Column                                                                          Non-Null Count  Dtype  
---  ------                                                                          --------------  -----  
 0   Total refunds from the Income Tax Department                                    286 non-null    float64
 1   Companies returns                                                               286 non-null    float64
 2   self employed returns                                                           286 non-null    float64
 3   Cancellations Deductions                                                        297 non-null    float64
 4   Capital Gas Tax Refunds                                                         296 non-null    float64
 5   Cancellation companies                                                          297 non-null    float64
 6  

## Correlation

In [4]:
import correlation_clustering

In [5]:
pairs = correlation_clustering.list_high_corr_pairs(merged_df, 0.75)

In [6]:
pairs

,var_i,var_j,rho,n_overlap
0,real_credit_card,Credit card purchase value dices (total) (2002...,0.999868,213
1,The combed dex of the state of the economy,madad_meshulav,0.996620,275
2,Net import purchase tax,Total import taxes,0.996417,286
3,Credit card purchase value dices (total) (2002...,madad_cc_purchases_sa,0.995408,214
4,real_credit_card,madad_cc_purchases_sa,0.995192,213
5,salaried jobs,salaried_jobs_total_sa,0.992724,202
6,TA125,TA35,0.980221,307
7,Total Income Tax Division Net,Total Gross Income Tax Division,0.964310,286
8,Total refunds from the Income Tax Department,Companies returns,0.963768,286
9,Real Salary,real_salary_avg_total_sa,0.955864,131


after evaluating these results I am applying these principals:
1. remove duplicates (I had several different sources of data)
2. keep the one with more availble records
3. pefer gross over net
4. do not remove economically related variables, these will be reduced to a factor together (example: real estate tax & praise tax, different income taxes etc.)

first items in table (0) are credit card purchases, that overalps almost completly, and then overlaps with the credit card index in item (4), since the cc index has significantly more records than the other two we can replace them.   
second item (1) overlaps almost completly with 'madad meshulav' which has more records.  
inspecting (5) and (9, 22-24) we can remove 'Real Salary' and 'salaried jobs'  
(7) remove 'Total Income Tax Division Net' and leave Gross  


In [8]:
merged_df = merged_df.drop(columns=merged_df.columns[[1, 12, 19, 20, 22, 26, 29, 48 ,51]])

In [9]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 308 entries, 1995-01-01 to 2025-04-01
Data columns (total 52 columns):
 #   Column                                                                          Non-Null Count  Dtype  
---  ------                                                                          --------------  -----  
 0   Total refunds from the Income Tax Department                                    286 non-null    float64
 1   self employed returns                                                           286 non-null    float64
 2   Cancellations Deductions                                                        297 non-null    float64
 3   Capital Gas Tax Refunds                                                         296 non-null    float64
 4   Cancellation companies                                                          297 non-null    float64
 5   Purchase returns                                                                296 non-null    float64
 6  

In [10]:
merged_df.to_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/EDA/merged_data.pkl')

In [11]:
pairs2 = correlation_clustering.list_high_corr_pairs(merged_df, 0.75)

In [12]:
pairs2

,var_i,var_j,rho,n_overlap
0,Employment rate (ages 25-64),"Participation rate (ages 25-64, percentage po...",0.888503,131
1,Real estate purchase tax,Real estate taxation,0.839921,286
2,praise tax,Real estate taxation,0.835030,286
3,Gross local VAT,Total net VAT,0.826101,286
4,Deduction from salary,Total Gross Income Tax Division,0.798940,286
5,Deductions and the capital market,Total Gross Income Tax Division,0.798893,286
6,Deduction from salary,Deductions and the capital market,0.778843,286
7,madad_pedion,madad_cc_purchases_sa,0.762170,295


## Proxy

Resample the mothly data to quarterly

In [13]:
quarterly_merged_df = merged_df.resample("QE").mean()

In [14]:
quarterly_merged_df.to_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/EDA/quarterly_merged_data.pkl')

In [15]:
quarterly_merged_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 122 entries, 1995-03-31 to 2025-06-30
Freq: QE-DEC
Data columns (total 52 columns):
 #   Column                                                                          Non-Null Count  Dtype  
---  ------                                                                          --------------  -----  
 0   Total refunds from the Income Tax Department                                    114 non-null    float64
 1   self employed returns                                                           114 non-null    float64
 2   Cancellations Deductions                                                        118 non-null    float64
 3   Capital Gas Tax Refunds                                                         118 non-null    float64
 4   Cancellation companies                                                          118 non-null    float64
 5   Purchase returns                                                                118 non-null   

In [ ]:
monthly_df.to_csv('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/formatted_CSVs/for_r/monthly_data.csv')

In [16]:
quarterly_df = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/stationary/quarterly_data.pkl')

In [17]:
quarterly_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 119 entries, 1995-06-30 to 2024-12-31
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   GDP     119 non-null    float64
 1   C       119 non-null    float64
 2   G       115 non-null    float64
 3   I       119 non-null    float64
 4   EX      119 non-null    float64
 5   IM      119 non-null    float64
dtypes: float64(6)
memory usage: 6.5 KB


In [ ]:
quarterly_df.to_csv('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/formatted_CSVs/for_r/quarterly_data.csv')

In [ ]:
quarterly_df.describe()

,GDP,C,G,I,EX,IM
count,119.000000,119.000000,115.000000,119.000000,119.000000,119.000000
mean,0.009132,0.009548,0.000037,0.007388,0.011957,0.010154
std,0.016473,0.021680,0.025058,0.062635,0.043540,0.040186
min,-0.086301,-0.122846,-0.133232,-0.306142,-0.177674,-0.114879
25%,0.002563,0.003015,-0.010309,-0.021964,-0.010088,-0.010720
50%,0.009972,0.009566,0.001903,0.008510,0.013298,0.013631
75%,0.015727,0.017629,0.009611,0.038962,0.036021,0.032206
max,0.085868,0.080171,0.159026,0.185486,0.165703,0.169621


In [18]:
# Align the dataframes by index
aligned_df = quarterly_merged_df.align(quarterly_df['GDP'], join='inner', axis=0)
quarterly_monthly_aligned = aligned_df[0]
quarterly_gdp_aligned = aligned_df[1]

# Calculate the correlation between GDP and each column in quarterly_monthly_df
correlations = quarterly_monthly_aligned.corrwith(quarterly_gdp_aligned)

# Find the column with the highest absolute correlation
most_correlated_column = correlations.abs().idxmax()
highest_correlation_value = correlations[most_correlated_column]

print(f"The column in quarterly_monthly_df with the highest correlation to GDP is: {most_correlated_column}")
print(f"The correlation coefficient is: {highest_correlation_value}")

The column in quarterly_monthly_df with the highest correlation to GDP is: salaried_jobs_total_sa
The correlation coefficient is: 0.5790178780404449


In [19]:
quarterly_monthly_aligned.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 119 entries, 1995-06-30 to 2024-12-31
Data columns (total 52 columns):
 #   Column                                                                          Non-Null Count  Dtype  
---  ------                                                                          --------------  -----  
 0   Total refunds from the Income Tax Department                                    112 non-null    float64
 1   self employed returns                                                           112 non-null    float64
 2   Cancellations Deductions                                                        116 non-null    float64
 3   Capital Gas Tax Refunds                                                         116 non-null    float64
 4   Cancellation companies                                                          116 non-null    float64
 5   Purchase returns                                                                116 non-null    float64
 6  

In [20]:
correlations = correlations.sort_values(ascending=False)

In [25]:
print(correlations[0:5])

salaried_jobs_total_sa                                0.579018
madad_meshulav                                        0.484566
Employment rate (ages 25-64)                          0.309802
Goods and services                                    0.219763
Participation rate (ages 25-64,  percentage pots)     0.160584
dtype: float64


In [26]:
import eda

In [29]:
quarterly_monthly_aligned = quarterly_monthly_aligned.join(quarterly_gdp_aligned)

In [37]:
eda.plot_time_series_altair(quarterly_monthly_aligned.iloc[0:, [49, 52, 47]], "Series Correlation With GDP Q")

alt.Chart(...)